<a href="https://colab.research.google.com/github/BerkeleyExpertSystemTechnologiesLab/Squishy-Methane-Analysis/blob/jberry/Squish_Robot_Quant_Model_v5_1_mm_%2B_image_transforms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Description

This model was produced by and for Squishy Robotics for the task of identifying and classifying methane leaks.


This model was made in conjunction with a synthetic dataset of 1 channel, 240 by 320 greyscale images of methane leaks
(1 x 240 x 320)


This model is experimental and uses the Optuna Hyperparameter Optimizer to search for successful hyperparameters (Learning Rate, Optimizer, Batch Size, Dropout %, etc...) and different optimizers. As such if you want to test a specific Model architecture you need to comment out the Optuna code and run a train/test on that specific model.

In [1]:
!pip install --quiet optuna # Hyperparameter Optimizer
!pip install --quiet timm   # Vision Transformer Library, pytorch compatible
# pip install transformers # Huggingface library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 36.4 MB/s eta 0:00:00


In [2]:
import os
import numpy as np

from collections import defaultdict
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import maxvit_t

from sklearn.model_selection import train_test_split
from torch.utils.data import random_split

# Vision Transformer Library
import timm

# Huggingface Transformer Library
import transformers

# Hyperparameter Search
import optuna

import json
import glob

#For file uploading
from google.colab import files
from google.colab import drive
from google.colab import auth


In [3]:
# This may take several minutes, the synthetic dataset can be large
#Upload the file
auth.authenticate_user()
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# This may take several minutes, the synthetic dataset can be large
!unzip -q "/content/drive/MyDrive/Squishy_Robotics_Dataset/Final_Dataset_single_channel_w_artif.zip" -d /content/

## Print out the shape of the data

In [5]:
# Loading an example file to demonstrate the dimensions
# This file might not exist, change the name to one that does to show the
# dimensions
file_path = './Final_Dataset_single_channel_w_artif/data/class_0/1237_frame_02_class_0.npy'
sample_data = np.load(file_path)
print(f"Shape of preprocessed sample data: {sample_data.shape}")
print(f"Data type of preprocessed sample data: {sample_data.dtype}")

# GasVid synthetic processed dataset should be 1 channels, 240x320 in dimension

Shape of preprocessed sample data: (1, 240, 320)
Data type of preprocessed sample data: float32


In [6]:
# Assuming the data is in 'Final_Dataset/data' and class folders are named 'class_0' ... 'class_7'
data_dir = 'Final_Dataset_single_channel_w_artif/data'
classes = sorted(os.listdir(data_dir))
print(f"Classes: {classes}")

Classes: ['class_0', 'class_1', 'class_2', 'class_3', 'class_4', 'class_5', 'class_6', 'class_7']


## Create a dataset and dataloader

In [7]:
class Multi_Modal_Dataset(Dataset):
    def __init__(self, numpy_files, json_files, labels, transform=None):
        """
        numpy_dir points to all the numpy 2 channel frames that were collected
          from METEC. This is designed to be 1st Channel Greyscale image of
          background, 2nd channel is just the gas plume scaled to some ppm
        json_dir points to all the metadata (ppm, distance, etc) that was
          collected from METEC or estimated using BEST Labs algorithms
        """
        self.numpy_files = numpy_files
        self.json_files = json_files
        self.labels = labels
        self.transform = transform


    def __len__(self):
      return len(self.numpy_files)


    def __getitem__(self, idx):
      numpy_path = self.numpy_files[idx]
      image_data = np.load(numpy_path)
      image_tensor = torch.from_numpy(image_data).float()

      if self.transform:
        image_tensor = self.transform(image_tensor)

      json_path = self.json_files[idx]
      with open(json_path, 'r') as f:
        metadata = json.load(f)

      metadat_features = self._extract_metadata_features(metadata)
      metadata_tensor = torch.tensor(metadat_features, dtype=torch.float32)

      label = self.labels[idx]

      return image_tensor, metadata_tensor, label


    def _extract_metadata_features(self, metadata):
      """
      Extracts a few entries from the metadata.
        For now:
          distance
          ppm
        In the future
          windspeed
          angle?
      """

      features = []

      # If the features exist, extract them, else place 0.0
      # Print warning statements if unable to retrieve the data
      distance = metadata.get("distance_m", None)
      if distance is None or distance == 0.0:
          print(f"WARNING: Invalid or missing distance_m value: {distance}")
          print(f"  Metadata keys available: {list(metadata.keys())}")
          features.append(0.0)
      else:
          features.append(distance)

      ppm = metadata.get("ppm", None)
      if ppm is None:
          print(f"WARNING: Missing ppm value")
          print(f"  Metadata keys available: {list(metadata.keys())}")
          features.append(0.0)
      else:
          features.append(ppm)

      return features

In [8]:
numpy_dir = "./Final_Dataset_single_channel_w_artif/data"
json_dir = "./Final_Dataset_single_channel_w_artif/metadata"

all_numpy_files = []
all_json_files = []
all_labels = []

print(f"Looking in: {numpy_dir}")
print(f"Directory exists: {os.path.exists(numpy_dir)}\n")

# Load each class separatley, collect the numpy and json files for a certain
# class at the same time
for class_idx in range(8):
    numpy_class_dir = os.path.join(numpy_dir, f"class_{class_idx}")
    json_class_dir = os.path.join(json_dir, f"class_{class_idx}")

    numpy_files_in_class = sorted(glob.glob(os.path.join(numpy_class_dir, "*.npy")))

    print(f"Class {class_idx}: Found {len(numpy_files_in_class)} files")

    for numpy_file in numpy_files_in_class:
        base_name = os.path.splitext(os.path.basename(numpy_file))[0]
        video_id = base_name.split('_')[0]


        json_filename = f"{video_id}_class_{class_idx}.json"
        json_file = os.path.join(json_class_dir, json_filename)

        if os.path.exists(json_file):
            all_numpy_files.append(numpy_file)
            all_json_files.append(json_file)
            all_labels.append(class_idx)
        else:
            print(f"WARNING: JSON missing for {base_name}")

print(f"\n{'='*60}")
print(f"TOTAL: {len(all_numpy_files)} numpy files")
print(f"TOTAL: {len(all_json_files)} json files")
print(f"{'='*60}\n")

# Only continue if we have files
if len(all_numpy_files) == 0:
    raise ValueError("!!!No files found!!! Check your paths above.")

# Now continue with video splitting
video_to_indices = defaultdict(list)
for idx, numpy_file in enumerate(all_numpy_files):
    video_id = os.path.basename(numpy_file).split('_')[0]
    video_to_indices[video_id].append(idx)

print(f"Number of unique videos: {len(video_to_indices)}")
print(f"Video IDs: {sorted(video_to_indices.keys())}\n")


Looking in: ./Final_Dataset_single_channel_w_artif/data
Directory exists: True

Class 0: Found 5391 files
Class 1: Found 5403 files
Class 2: Found 5410 files
Class 3: Found 5393 files
Class 4: Found 5421 files
Class 5: Found 5412 files
Class 6: Found 5394 files
Class 7: Found 5395 files

TOTAL: 43219 numpy files
TOTAL: 43219 json files

Number of unique videos: 28
Video IDs: ['1237', '1238', '1239', '1240', '1241', '1242', '1467', '1468', '1469', '1470', '1471', '1472', '2559', '2560', '2561', '2562', '2563', '2564', '2566', '2567', '2568', '2569', '2571', '2578', '2579', '2580', '2581', '2583']



In [9]:
video_to_indices = defaultdict(list) #Make an empty dictionary of lists

for idx, numpy_file in enumerate(all_numpy_files):
    video_id = os.path.basename(numpy_file).split('_')[0] #Extract 4 digit code from numpy filename
    video_to_indices[video_id].append(idx)

video_ids = list(video_to_indices.keys())

# Split the video into train and test
train_vids, test_vids = train_test_split(video_ids, test_size=0.2, random_state=42)

# Verify no overlap
overlap = set(train_vids) & set(test_vids)
if overlap:
    print(f"\nVideos overlap: {overlap}")
else:
    print(f"\nNo video overlap - train and test are separate")

train_indices = []
test_indices = []

for vid in train_vids:
    train_indices.extend(video_to_indices[vid])
for vid in test_vids:
    test_indices.extend(video_to_indices[vid])

# Create file lists
train_numpy = [all_numpy_files[i] for i in train_indices]
train_json = [all_json_files[i] for i in train_indices]
train_labels_list = [all_labels[i] for i in train_indices]

test_numpy = [all_numpy_files[i] for i in test_indices]
test_json = [all_json_files[i] for i in test_indices]
test_labels_list = [all_labels[i] for i in test_indices]



No video overlap - train and test are separate


## Image Transformations

In [10]:
# Augmentation section
# https://docs.pytorch.org/vision/0.13/transforms.html
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1),
    ),
    transforms.RandomApply([
        transforms.GaussianBlur(
            kernel_size=3,
            sigma=(0.1, 2.0)
        )
    ], p=0.3)
])

#During testing don't use augmentation
test_transforms = transforms.Compose([
    transforms.Resize((224, 224))
])

In [11]:
# SHOW FINAL SPLIT STATISTICS
print(f"\n{'='*60}")
print("DATASET STATISTICS")
print("="*90)

print(f"\nTRAINING SET:")
print(f"   Total samples: {len(train_numpy)}")
print(f"   From {len(train_vids)} videos: {sorted(train_vids)}")

# Count samples per class in training
train_class_counts = Counter(train_labels_list)
print(f"\n   Samples per class:")
for class_id in range(8):
    count = train_class_counts.get(class_id, 0)
    percentage = (count / len(train_numpy) * 100) if len(train_numpy) > 0 else 0
    print(f"      Class {class_id}: {count:5d} samples ({percentage:5.2f}%)")

print(f"\nTEST SET:")
print(f"   Total samples: {len(test_numpy)}")
print(f"   From {len(test_vids)} videos: {sorted(test_vids)}")

# Count samples per class in testing
test_class_counts = Counter(test_labels_list)
print(f"\n   Samples per class:")
for class_id in range(8):
    count = test_class_counts.get(class_id, 0)
    percentage = (count / len(test_numpy) * 100) if len(test_numpy) > 0 else 0
    print(f"      Class {class_id}: {count:5d} samples ({percentage:5.2f}%)")

# VERIFY ALL CLASSES PRESENT
print(f"\n{'='*70}")
print("VERIFICATION")
print("="*70)

train_classes = set(train_labels_list)
test_classes = set(test_labels_list)
missing_train = set(range(8)) - train_classes
missing_test = set(range(8)) - test_classes

if missing_train:
    print(f"WARNING: Training missing classes {missing_train}")
else:
    print(f"Training set has all 8 classes")

if missing_test:
    print(f"WARNING: Testing missing classes {missing_test}")
else:
    print(f"Test set has all 8 classes")

# Show train/test split ratio
total_samples = len(train_numpy) + len(test_numpy)
train_ratio = len(train_numpy) / total_samples * 100
test_ratio = len(test_numpy) / total_samples * 100
print(f"\nSplit ratio: {train_ratio:.1f}% train / {test_ratio:.1f}% test")

print(f"\n{'='*70}")
print("DATA SPLIT COMPLETE AND VERIFIED")
print("="*70)



DATASET STATISTICS

TRAINING SET:
   Total samples: 33973
   From 22 videos: ['1238', '1239', '1240', '1241', '1242', '1467', '1468', '1471', '1472', '2560', '2561', '2562', '2563', '2564', '2566', '2567', '2568', '2571', '2578', '2579', '2581', '2583']

   Samples per class:
      Class 0:  4236 samples (12.47%)
      Class 1:  4258 samples (12.53%)
      Class 2:  4250 samples (12.51%)
      Class 3:  4239 samples (12.48%)
      Class 4:  4260 samples (12.54%)
      Class 5:  4241 samples (12.48%)
      Class 6:  4241 samples (12.48%)
      Class 7:  4248 samples (12.50%)

TEST SET:
   Total samples: 9246
   From 6 videos: ['1237', '1469', '1470', '2559', '2569', '2580']

   Samples per class:
      Class 0:  1155 samples (12.49%)
      Class 1:  1145 samples (12.38%)
      Class 2:  1160 samples (12.55%)
      Class 3:  1154 samples (12.48%)
      Class 4:  1161 samples (12.56%)
      Class 5:  1171 samples (12.66%)
      Class 6:  1153 samples (12.47%)
      Class 7:  1147 samples

In [12]:
train_dataset = Multi_Modal_Dataset(train_numpy,
                                    train_json,
                                    train_labels_list,
                                    transform=train_transforms)
test_dataset = Multi_Modal_Dataset(test_numpy,
                                   test_json,
                                   test_labels_list,
                                   transform=test_transforms)

# Define DeiT ViT model

## Define the Optuna Objective Function

This function will be called by Optuna for each trial. It will:
1. Suggest hyperparameters using the trial object.
2. Build and train the Swin ViT model with the suggested hyperparameters.
3. Evaluate the model on a validation set
4. Return the metric to minimize (loss) or maximize (accuracy).

In [13]:
def build_maxvit_backbone(in_channels=1):
    model = maxvit_t(weights=None)
    # Stem: first conv is model.stem[0]; first submodule is Conv2d
    old = model.stem[0][0]
    model.stem[0][0] = nn.Conv2d(
        in_channels,
        old.out_channels,
        kernel_size=old.kernel_size,
        stride=old.stride,
        padding=old.padding,
        bias=False,
    )
    # Output feature vector (B, 512) instead of logits; maxvit_t last block has 512 channels
    model.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
    return model, 512

In [14]:
def objective(trial):

    #############################
    # All Hyperparameters Tested
    #############################
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'AdamW'])
    momentum = trial.suggest_float('momentum', 0.0, 0.99) if optimizer_name in ['SGD'] else 0.0
    weight_decay = trial.suggest_float('weight_decay', 0.0, 0.01)
    # Switched to smaller batch sizes to avoid  running out of memory
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32, 64])
    num_epochs = trial.suggest_int('num_epochs', 10, 25)
    fc_drop_rate = trial.suggest_float('fc_drop_rate', 0.2, 0.6)

    #####################
    # Define the Model
    #####################
    class MultiModeMaxViT(nn.Module):
        def __init__(self, num_classes=8, in_channels=1, num_metadata_feats=2, fc_drop_rate=0.3):
            super(MultiModeMaxViT, self).__init__()
            self.maxvit, num_features = build_maxvit_backbone(in_channels=in_channels)
            self._maxvit_features = num_features

            #Smaller Neural Net for metadata only
            self.metadata_fc = nn.Sequential(
                nn.Linear(num_metadata_feats, 64),
                nn.ReLU(),
                nn.Dropout(fc_drop_rate),
                nn.Linear(64,64)
            )

            #Classifier combines metadata NN and image Swin ViT outputs
            self.classifier = nn.Sequential(
                nn.Linear(num_features + 64, 128),
                nn.ReLU(),
                nn.Dropout(fc_drop_rate),
                nn.Linear(128, num_classes)
            )

        def forward(self, image, metadata):
            maxvit_out = self.maxvit(image)
            meta_out = self.metadata_fc(metadata)
            combined = torch.cat([maxvit_out, meta_out], dim=1)
            return self.classifier(combined)

    model = MultiModeMaxViT(
        num_classes=8,
        in_channels=1,
        num_metadata_feats=2,
        fc_drop_rate=fc_drop_rate
    )

    ###############################
    # Define optimizer and scaler for mixed precision
    ###############################
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer name: {optimizer_name}")

    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler() # Initialize GradScaler for mixed precision

    ##########################################
    # Create DataLoaders with trial batch_size
    ##########################################
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    ###############################
    # Train the model
    ###############################
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)


    print(f"\n{'='*70}")
    print(f"Trial {trial.number} | lr={lr:.6f} | optimizer={optimizer_name} | "
          f"batch={batch_size} | epochs={num_epochs} | fc_drop={fc_drop_rate:.3f}")
    print(f"{'='*70}")


    model.train()
    train_correct = 0
    train_total = 0
    for epoch in range(num_epochs):
        train_correct = 0
        train_total = 0
        train_loss = 0.0
        num_batches = 0

        for images, metadata, labels in train_loader:
            images = images.to(device)
            metadata = metadata.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            # Use autocast for mixed precision training ( going from float 32 to float 16)
            with torch.cuda.amp.autocast():
                outputs = model(images, metadata)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward() # Scale loss and call backward()
            scaler.step(optimizer) # Update optimizer
            scaler.update() # Update scaler for next iteration

            # Calculate loss
            train_loss += loss.item()
            num_batches += 1

            # Calculate training accuracy
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        #Print out training during each epoch
        train_accuracy = train_correct / train_total
        avg_train_loss = train_loss / num_batches

        # Print training accuracy for this epoch
        print(f"Epoch [{epoch+1:2d}/{num_epochs}] Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.4f}")

    #####################
    # Evaluate the model
    #####################
    model.eval()
    correct, total = 0, 0
    val_loss = 0.0
    num_val_batches = 0
    with torch.no_grad():
        for images, metadata, labels in test_loader:
            images = images.to(device)
            metadata = metadata.to(device)
            labels = labels.to(device)

            # Use autocast for mixed precision inference
            with torch.cuda.amp.autocast():
                outputs = model(images, metadata)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            num_val_batches += 1

            # Calculate Validation Accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    avg_val_loss = val_loss / num_val_batches

    print(f"Validation Loss: {avg_val_loss:.4f} | Validation Acc: {accuracy:.4f}")
    print(f"{'='*70}\n")

    # Clear GPU cache after each trial to prevent running out of mem
    del model, optimizer, scaler, train_loader, test_loader, images, metadata, labels, outputs
    torch.cuda.empty_cache()

    return accuracy

## Run the Optuna Study

Create an Optuna study and run the optimization process.

In [15]:
# Create a study object and specify the direction of optimization (maximize accuracy)
study = optuna.create_study(direction='maximize',
                             pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))

# Run the optimization
study.optimize(objective, n_trials = 10)

# Print the best hyperparameters found
print("Best hyperparameters: ", study.best_params)

# Print the best accuracy found
print("Best accuracy: ", study.best_value)

# Plot the visualization
optuna.visualization.plot_param_importances(study).show()

# Run more trials
# study.optimize(objective, n_trials=20)

[I 2026-02-06 21:15:15,711] A new study created in memory with name: no-name-12305bf1-778f-4eeb-b730-5186ab8013b5
/tmp/ipython-input-2169940886.py:66: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # Initialize GradScaler for mixed precision



Trial 0 | lr=0.001538 | optimizer=AdamW | batch=16 | epochs=13 | fc_drop=0.426


/tmp/ipython-input-2169940886.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [ 1/13] Train Loss: 2.0883 | Train Acc: 0.1265
Epoch [ 2/13] Train Loss: 2.3243 | Train Acc: 0.1218
Epoch [ 3/13] Train Loss: 2.0797 | Train Acc: 0.1254
Epoch [ 4/13] Train Loss: 2.0798 | Train Acc: 0.1254
Epoch [ 5/13] Train Loss: 2.0798 | Train Acc: 0.1233
Epoch [ 6/13] Train Loss: 2.0798 | Train Acc: 0.1209
Epoch [ 7/13] Train Loss: 2.0798 | Train Acc: 0.1231
Epoch [ 8/13] Train Loss: 2.0797 | Train Acc: 0.1235
Epoch [ 9/13] Train Loss: 2.0798 | Train Acc: 0.1207
Epoch [10/13] Train Loss: 2.0798 | Train Acc: 0.1230
Epoch [11/13] Train Loss: 2.0798 | Train Acc: 0.1229
Epoch [12/13] Train Loss: 2.0797 | Train Acc: 0.1243
Epoch [13/13] Train Loss: 2.0798 | Train Acc: 0.1225


/tmp/ipython-input-2169940886.py:141: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[I 2026-02-06 22:39:41,414] Trial 0 finished with value: 0.12491888384166126 and parameters: {'lr': 0.001538093506042329, 'optimizer': 'AdamW', 'weight_decay': 0.003927952094675506, 'batch_size': 16, 'num_epochs': 13, 'fc_drop_rate': 0.42597334825132094}. Best is trial 0 with value: 0.12491888384166126.


...

Validation Loss: 1.1936 | Validation Acc: 0.4767


Trial 8 | lr=0.026823 | optimizer=AdamW | batch=8 | epochs=25 | fc_drop=0.557
Epoch [ 1/25] Train Loss: nan | Train Acc: 0.1251
Epoch [ 2/25] Train Loss: nan | Train Acc: 0.1224
Epoch [ 3/25] Train Loss: nan | Train Acc: 0.1253
Epoch [ 4/25] Train Loss: nan | Train Acc: 0.1241
Epoch [ 5/25] Train Loss: nan | Train Acc: 0.1266
Epoch [ 6/25] Train Loss: nan | Train Acc: 0.1240
Epoch [ 7/25] Train Loss: nan | Train Acc: 0.1247
Epoch [ 8/25] Train Loss: nan | Train Acc: 0.1229
Epoch [ 9/25] Train Loss: nan | Train Acc: 0.1230
Epoch [10/25] Train Loss: nan | Train Acc: 0.1260
Epoch [11/25] Train Loss: nan | Train Acc: 0.1251
Epoch [12/25] Train Loss: nan | Train Acc: 0.1267
Epoch [13/25] Train Loss: nan | Train Acc: 0.1247
Epoch [14/25] Train Loss: nan | Train Acc: 0.1247
Epoch [15/25] Train Loss: nan | Train Acc: 0.1247
Epoch [16/25] Train Loss: nan | Train Acc: 0.1247
Epoch [17/25] Train Loss: nan | Train Acc: 0.1247
Epoch [18/25] Train L

[I 2026-02-07 14:51:35,144] Trial 8 finished with value: 0.12491888384166126 and parameters: {'lr': 0.026823264308876184, 'optimizer': 'AdamW', 'weight_decay': 0.001008773170659083, 'batch_size': 8, 'num_epochs': 25, 'fc_drop_rate': 0.5572143088739774}. Best is trial 4 with value: 0.5248756218905473.


Validation Loss: nan | Validation Acc: 0.1249


Trial 9 | lr=0.001225 | optimizer=SGD | batch=32 | epochs=22 | fc_drop=0.212
Epoch [ 1/22] Train Loss: 1.8216 | Train Acc: 0.2652
Epoch [ 2/22] Train Loss: 1.5170 | Train Acc: 0.3671
Epoch [ 3/22] Train Loss: 1.3635 | Train Acc: 0.4133
Epoch [ 4/22] Train Loss: 1.2815 | Train Acc: 0.4419
Epoch [ 5/22] Train Loss: 1.1975 | Train Acc: 0.4769
Epoch [ 6/22] Train Loss: 1.1252 | Train Acc: 0.5058
Epoch [ 7/22] Train Loss: 1.0756 | Train Acc: 0.5237
Epoch [ 8/22] Train Loss: 1.0296 | Train Acc: 0.5436
Epoch [ 9/22] Train Loss: 0.9858 | Train Acc: 0.5631
Epoch [10/22] Train Loss: 0.9469 | Train Acc: 0.5772
Epoch [11/22] Train Loss: 0.9172 | Train Acc: 0.5924
Epoch [12/22] Train Loss: 0.8937 | Train Acc: 0.6038
Epoch [13/22] Train Loss: 0.8620 | Train Acc: 0.6167
Epoch [14/22] Train Loss: 0.8392 | Train Acc: 0.6340
Epoch [15/22] Train Loss: 0.8158 | Train Acc: 0.6398
Epoch [16/22] Train Loss: 0.7856 | Train Acc: 0.6523
Epoch [17/22] Train Loss: 0

[I 2026-02-07 16:26:20,676] Trial 9 finished with value: 0.3538827601124811 and parameters: {'lr': 0.0012246709288056103, 'optimizer': 'SGD', 'momentum': 0.11741647107992825, 'weight_decay': 0.0006129234841941667, 'batch_size': 32, 'num_epochs': 22, 'fc_drop_rate': 0.21154320205118374}. Best is trial 4 with value: 0.5248756218905473.


Validation Loss: 2.3727 | Validation Acc: 0.3539

Best hyperparameters:  {'lr': 0.0006621560448688664, 'optimizer': 'SGD', 'momentum': 0.7860127605621317, 'weight_decay': 0.008979843434144505, 'batch_size': 16, 'num_epochs': 10, 'fc_drop_rate': 0.4280007391235857}
Best accuracy:  0.5248756218905473


# Sources:
### Hyperparameter Tuning with Optuna:
https://medium.com/@taeefnajib/hyperparameter-tuning-using-optuna-c46d7b29a3e
https://optuna.org/#code_examples

### Multi-Modal ML Models
https://www.nature.com/articles/s41598-025-14901-4
https://www.reddit.com/r/MachineLearning/comments/nziumg/combining_images_and_other_numeric_features_in_a/
https://pyimagesearch.com/2019/02/04/keras-multiple-inputs-and-mixed-data/

### ViT Models
https://www.geeksforgeeks.org/deep-learning/building-a-vision-transformer-from-scratch-in-pytorch/

https://www.youtube.com/watch?v=7o1jpvapaT0&t=2924s

https://medium.com/correll-lab/building-a-vision-transformer-model-from-scratch-a3054f707cc6
